In [ ]:
from src.config import AUTH_TOKEN_HF
AUTH_TOKEN_HF


In [ ]:
from src.get_dataset_info import get_all_info
df_transcriptions, df_results, df_embeddings, df_features = get_all_info(transcribe=False, evaluate=False, embeddings=False, features= False)

try:
    df_features = df_features.drop(columns=['label'])
except KeyError:
    pass

In [ ]:
import os
df = df_results.merge(df_embeddings, on="id", how="inner")
dataset = df.merge(df_features, on="id", how="inner")
print("Final DataFrame shape:", dataset.shape)

In [ ]:
X = dataset.drop(columns=['label'])
y = dataset['label'].values

In [ ]:
import pandas as pd
import numpy as np
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

enet_cv = LogisticRegressionCV(
    penalty='elasticnet',
    solver='saga',
    l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9],
    Cs=np.logspace(-3, 2, 20),
    cv=5,
    scoring='f1',
    max_iter=5000,
    n_jobs=-1
)

enet_cv.fit(X_scaled, y)

print("Best C:", enet_cv.C_[0])
print("Best l1_ratio:", enet_cv.l1_ratio_[0])

importance_enet = np.mean(np.abs(enet_cv.coef_), axis=0)

df_enet = pd.DataFrame({
    "feature": X.columns,
    "score": importance_enet
}).sort_values("score", ascending=False)

df_enet[df_enet["score"] > 0].head(10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
df_enet_plot_10 = df_enet[df_enet["score"] > 0].head(10)

df_plot = df_enet_plot_10.sort_values("score", ascending=True)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=df_plot,
    x="score",
    y="feature",
    palette="viridis"
)

plt.title("Top 10 features más relevantes (Elastic Net)")
plt.xlabel("Importancia (|coeficiente| medio)")
plt.ylabel("Feature")

plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
import os
df_importancia = df_enet[df_enet["score"] > 0]

count_features_clinicas = 0
count_embeddings_audio = 0
count_embedding_linguistic = 0

list_features_clinicas = []
list_embeddings_audio = []
list_embedding_linguistic = []
list_otros = []
for index,row in df_importancia.iterrows():
    feature = row['feature']

    if feature.startswith('score') or feature.startswith('count_'):
        count_features_clinicas += 1
        list_features_clinicas.append(feature)
    elif feature.startswith('embedding_audio'):
        count_embeddings_audio += 1
        list_embeddings_audio.append(feature)
    elif feature.startswith('embedding_text'):
        count_embedding_linguistic += 1
        list_embedding_linguistic.append(feature)
    else:
        list_otros.append(feature)

print(f"Selected clinical features: {len(list_features_clinicas)} | {list_features_clinicas}")
print(f"Embeddings de audio seleccionados: {len(list_embeddings_audio)} | {list_embeddings_audio}")
print(f"Embeddings lingüísticos seleccionados: {len(list_embedding_linguistic)} | {list_embedding_linguistic}")
print(f"Selected remaining features: {len(list_otros)} | {list_otros}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
df_enet["type"] = df_enet["feature"].apply(
    lambda x: (
        "Clinical" if x.startswith("score") else
        "Audio embedding" if "embedding_audio" in x else
        "Text embedding" if "embedding_text" in x else
        "Acústica" if any(k in x for k in ["waveform", "energia", "speaking", "pronoun"]) else
        "Otros"
    )
)
df_plot = df_enet[df_enet["score"] > 0].copy()
df_plot = df_plot.sort_values("score", ascending=True)

plt.figure(figsize=(10, 12))

sns.barplot(
    data=df_plot,
    x="score",
    y="feature",
    hue="type",
    dodge=False
)

plt.title("Selected feature importance (Elastic Net)")
plt.xlabel("Importancia (|coeficiente|)")
plt.ylabel("Feature")

plt.legend(title="Tipo de feature")
plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
features_enet = df_enet[df_enet["score"] > 0]["feature"].tolist()
print("Features selected by Elastic Net:", features_enet)

X_enet = X[features_enet].copy()
y = df['label'].values

mi = mutual_info_classif(X_enet, y)

df_mi = pd.DataFrame({
    "feature": X_enet.columns,
    "mi_score": mi
}).sort_values("mi_score", ascending=False)

top_feats_mRMR = df_mi["feature"].tolist()[:20]

print("Top features por Mutual Information:", top_feats_mRMR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
features_enet = df_enet[df_enet["score"] > 0]["feature"].tolist()
print("Features selected by Elastic Net:", features_enet)

palette = {
    "CN": sns.color_palette("Set2")[0],
    "AD": sns.color_palette("Set2")[1]
}

X_sel = X[features_enet].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sel)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    'Diagnosis': ['AD' if label == 1 else 'CN' for label in dataset["label"].values]
})

plt.figure(figsize=(8, 6))

groups = df_pca["Diagnosis"].unique()

for g in groups:
    subset = df_pca[df_pca["Diagnosis"] == g]
    plt.scatter(subset["PC1"], subset["PC2"], label=g, alpha=0.7, color=palette.get(g, "gray"))

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("PCA on selected features (Elastic Net)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
palette = {
    "CN": sns.color_palette("Set2")[0],
    "AD": sns.color_palette("Set2")[1]
}

X_sel = X[features_enet].copy()

X_scaled = StandardScaler().fit_transform(X_sel)

tsne = TSNE(
    n_components=2,
    perplexity=50,   # importante
    learning_rate=200,
    random_state=42
)

X_tsne = tsne.fit_transform(X_scaled)

df_tsne = pd.DataFrame({
    "TSNE1": X_tsne[:, 0],
    "TSNE2": X_tsne[:, 1],
    "label": ["AD" if label == 1 else "CN" for label in dataset["label"].values]
})

plt.figure(figsize=(8, 6))

for g in df_tsne["label"].unique():
    subset = df_tsne[df_tsne["label"] == g]
    plt.scatter(subset["TSNE1"], subset["TSNE2"], label=g, alpha=0.7, color=palette.get(g, "gray"))

plt.title("t-SNE on selected features")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()